# Part 7 · Notebook 05 — The option strategy builder

**Sessions:** S5 (Option strategy builder) · [Lesson plan](../../docs/lessons/PART_07_STRATEGY_LIBRARY.md) · graded labs in [`labs/part07/`](../../labs/part07/)

**You will:**
1. Value a multi-leg strategy today, later, and at expiry.
2. Find the breakevens of its expiry P&L.
3. Decide whether a structure is defined-risk.
4. See why a high probability of profit is not an edge.

How these notebooks work: the setup, data and plotting code is written for you. Cells marked **✍️ Your turn** need a few lines from you.
If your answer does not match yet, the notebook continues with the reference answer so nothing else breaks.
All data is synthetic, built from regimes you know, and every strategy here is a **hypothesis** with a first-look evaluation: the honest backtest comes in Part 8.

In [ ]:
import sys
from pathlib import Path
for d in (Path.cwd(), Path.cwd().parent):       # p7lib.py is in notebooks/part07/
    sys.path.insert(0, str(d))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import p7lib as p

p.use_course_style()

## 1. Legs and strategies

A strategy is a list of legs `(cp, K, T, qty, iv)` with `cp = +1` call, `−1` put, `0` for the underlying, and a multiplier of 100. Here is a 30-day iron condor on a stock at 600, shorts at about 16 delta, wings 10 points further out, each strike priced on a simple equity smile (`p.skew_iv`).

In [ ]:
S, T = 600.0, 30 / 365
ivf = p.skew_iv(S)
strikes = np.arange(450, 751, 5.0)
ic = p.iron_condor(S, T, ivf, strikes)
pd.DataFrame([vars(L) for L in ic.legs]).round(4)

## 2. The value of a strategy

Mark each leg after `dt` years have passed and with every IV shifted by `dvol`: an option leg with time left (`tau = T − dt > 1e-9`) at `p.bsm_price(S, K, tau, r, q, iv + dvol, cp)`, an expired one at its intrinsic value `max(cp·(S − K), 0)`, the underlying at `S`. Sum `qty × value` and multiply by the multiplier. `S` may be an array.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def strategy_value(strat, S, r=0.04, q=0.0, dt=0.0, dvol=0.0):
    S = np.asarray(S, dtype=float)
    v = np.zeros_like(S)
    for L in strat.legs:
        if L.cp == 0:
            v = v + L.qty * S
            continue
        tau = L.T - dt
        leg = ...                                 # ✍️ BSM with the time left, or intrinsic value once expired
        v = v + L.qty * leg
    return v * strat.multiplier

grid = np.linspace(540, 660, 7)
cases = [dict(dt=0.0), dict(dt=15 / 365), dict(dt=T), dict(dt=0.0, dvol=0.05)]
mine = [p.attempt(strategy_value, ic, grid, **k) for k in cases]
mine = p.check("strategy_value", mine, [ic.value(grid, **k) for k in cases])
pd.DataFrame(np.round(mine, 0), index=["today", "in 15 days", "at expiry", "today, IV +5"], columns=grid)

The condor is a **credit** (negative value today: we receive money). It gains as time passes and loses if IV rises.

In [ ]:
g = np.linspace(520, 680, 400)
cost = float(ic.value(S))
fig, ax = plt.subplots()
for dt, lab in [(0.0, "today"), (15 / 365, "in 15 days"), (T, "at expiry")]:
    ax.plot(g, ic.value(g, dt=dt) - cost, label=lab)
ax.axhline(0, color="black", lw=0.6); ax.axvline(S, color="#e6e5e0")
ax.set(xlabel="stock price", ylabel="P&L, $", title="Iron condor P&L"); ax.legend(); plt.show()
ic.greeks(S)

## 3. Breakevens

On a fine price grid, the breakevens are where the expiry P&L changes sign: take `grid[1:]` where `sign(pnl[1:]) != sign(pnl[:-1])`, rounded to 2 decimals.

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def breakevens(grid, pnl):
    sign = np.sign(pnl)
    return ...                                    # ✍️

gg = np.linspace(0.5 * S, 1.5 * S, 2001)
pnl = ic.payoff_at_first_expiry(gg) - ic.value(S)
mine = p.attempt(breakevens, gg, pnl)
mine = p.check("breakevens", mine, ic.analyze(S)["breakevens"])
a = ic.analyze(S)
print(f"breakevens {mine}; credit ${-a['cost']:.0f}; max profit ${a['max_profit']:.0f}; max loss ${a['max_loss']:.0f}; POP {a['pop']:.0%}")

## 4. A high POP is not an edge

The condor wins about 70% of the time, but its maximum loss is nearly four times its maximum profit. Simulate many expiries under the same one-volatility lognormal the POP assumes and look at the average: a 71% win rate and still a **losing** trade on average, because each loss is worth three wins. Win rate tells you how the P&L is *shaped*, not whether it's positive. Size short-premium trades by their max loss, not their POP.

In [ ]:
rng = np.random.default_rng(0)
sig = np.mean([L.iv for L in ic.legs])
ST = S * np.exp((0.04 - 0.5 * sig ** 2) * T + sig * np.sqrt(T) * rng.standard_normal(200_000))
sim = ic.payoff_at_first_expiry(ST) - ic.value(S) * np.exp(0.04 * T)
print(f"win rate {np.mean(sim > 0):.0%}, average P&L ${sim.mean():+.1f}, average win ${sim[sim > 0].mean():.0f}, average loss ${sim[sim <= 0].mean():.0f}")

## 5. Defined risk

The library's templates are **defined-risk** by default: every short option is covered. Rule: net call quantity plus shares must be `>= 0` (short calls covered by long calls or stock), and net put quantity `>= 0` (short puts covered by long puts).

✍️ **Your turn** — replace each `...` and run the cell. `p.check` tells you if you are right.

In [ ]:
def is_defined_risk(strat):
    calls = sum(L.qty for L in strat.legs if L.cp == 1)
    puts = sum(L.qty for L in strat.legs if L.cp == -1)
    shares = sum(L.qty for L in strat.legs if L.cp == 0)
    return ...                                    # ✍️

book = {"iron condor": ic,
        "short strangle": p.OptionStrategy("short strangle").add(-1, 560, T, -1, ivf(560)).add(1, 640, T, -1, ivf(640)),
        "covered call": p.OptionStrategy("covered call").add(0, 0, 0, 1).add(1, 630, T, -1, ivf(630)),
        "put ratio spread": p.OptionStrategy("1×2 put ratio").add(-1, 590, T, 1, ivf(590)).add(-1, 570, T, -2, ivf(570)),
        "bull call spread": p.vertical(1, 600, 620, T, ivf)}
mine = {k: p.attempt(is_defined_risk, v) for k, v in book.items()}
mine = p.check("is_defined_risk", mine, {k: v.is_defined_risk() for k, v in book.items()})
mine

## Wrap-up

* One builder values any structure today, later, at expiry and under vol shifts.
* Report breakevens, max profit, max loss and POP, and never size by POP.
* Templates default to defined risk.
* Graded version: `labs/part07/week24_option_builder` (the builder, templates, IB `BAG` and Alpaca multi-leg orders).